# 09 - M9: main experiment, steered vs plain (real CLIP, real T4)
**Runtime > T4 GPU.** Gate (contract): both arms complete, results JSON written.
Frozen settings (D022): fold Midjourney, `controlled` protocol, 5 epochs, batch 8, `TrainConfig` defaults. The arms differ in `arm` only.

**Time:** about 45-50 min per run (estimated from M8's 6.4 img/s), so about 1.5-2 h for seed 0.
**Disconnected?** Run all cells again. Finished runs are skipped, the unfinished one resumes from its checkpoint on Drive.
**Do not** change any setting after a test result has been printed, and do not delete a run folder to run it again.

In [ ]:
# BOOTSTRAP - identical first cell in every notebook. Safe to re-run: clones the repo if missing, otherwise pulls.
import os, subprocess
from google.colab import drive, userdata
drive.mount('/content/drive')
DRIVE_ROOT = '/content/drive/MyDrive/csm-clip'
os.makedirs(DRIVE_ROOT + '/runs', exist_ok=True)
GH_USER = 'bhargav-reddy-nasappagari'
try:
    tok = userdata.get('GITHUB_TOKEN')      # only needed if the repo is private
except Exception:
    tok = None
url = f"https://{tok + '@' if tok else ''}github.com/{GH_USER}/csm-clip.git"
if not os.path.exists('/content/csm-clip'):
    subprocess.run(['git', 'clone', '-q', url, '/content/csm-clip'], check=True)
else:
    subprocess.run(['git', '-C', '/content/csm-clip', 'pull', '-q', '--rebase', url, 'master'], check=True)
os.chdir('/content/csm-clip')
subprocess.run(['git', 'remote', 'set-url', 'origin', f'https://github.com/{GH_USER}/csm-clip.git'], check=True)  # keep token out of .git/config
subprocess.run(['pip', 'install', '-q', '-r', 'requirements.txt'], check=True)
print(subprocess.run(['git', 'log', '--oneline', '-3'], capture_output=True, text=True).stdout)

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!python -m pytest -q

In [ ]:
import json, gc, time, numpy as np, torch
from datasets import load_dataset
from transformers import CLIPVisionModel
from src.data import load_manifest, ManifestDataset
from src import concepts as K
from src import experiment as E
from src.train import make_model, predict
dev = 'cuda'
SEEDS = [0]          # gate needs seed 0. Later: [0, 1, 2] and run all again (seed 0 is skipped, not repeated)
RUNS = f'{DRIVE_ROOT}/runs/m9'; os.makedirs(RUNS, exist_ok=True)   # never wiped: it holds the checkpoints and results
assert os.path.exists('assets/concepts_text_768.npy'), 'concept embeddings missing from the repo'
T = K.load_or_encode('assets/concepts_text_768.npy')
mf = load_manifest(E.M9_BASE['fold'])
ds = load_dataset('TheKernel01/Tiny-GenImage', revision=mf['revision'])
split = lambda which: ManifestDataset(mf, which, ds['train'], ds['validation'], protocol=E.M9_BASE['protocol'])
train_ds, val_ds, test_ds = split('train'), split('val'), split('test')   # train flips, val/test do not
print('train/val/test:', len(train_ds), len(val_ds), len(test_ds), '| held out', mf['held_out'], '| seen', mf['seen'])
assert (len(train_ds), len(val_ds), len(test_ds)) == (3240, 360, 1000)

def build_model(cfg):
    clip = CLIPVisionModel.from_pretrained(K.TEXT_CHECKPOINT, torch_dtype=torch.float32).eval()
    return make_model(clip, T, cfg)

In [ ]:
# PRE-FLIGHT (about 2 min, validation only). The M8 gates never ran prediction on the real model, and "matched" is
# worth checking on the real backbone: same shared weights, identical logits at init, gates the only extra parameters.
from torch.utils.data import Subset
small = Subset(val_ds, list(range(32)))
out, sd, rep = {}, {}, {}
for arm in E.ARM_ORDER:
    m = build_model(E.m9_config(arm, SEEDS[0])).to(dev)
    torch.cuda.reset_peak_memory_stats(); t0 = time.time()
    out[arm], y, _ = predict(m, small, batch_size=16, device=dev, amp=True, num_workers=2)
    print(f'{arm}: predict 32 imgs {time.time()-t0:.1f}s, peak {torch.cuda.max_memory_allocated()/2**30:.2f} GiB', m.param_report())
    sd[arm], rep[arm] = m.trainable_state_dict(), m.param_report()
    del m; gc.collect(); torch.cuda.empty_cache()
E.check_matched(E.m9_config('steered', SEEDS[0]), E.m9_config('plain', SEEDS[0]))
assert all(torch.equal(v, sd['steered'][k]) for k, v in sd['plain'].items()), 'shared initial weights differ'
assert all('.gates.' in k for k in set(sd['steered']) - set(sd['plain'])), 'arms differ in more than the gates'
assert rep['steered']['trainable'] - rep['plain']['trainable'] == rep['steered']['gates']
diff = float(np.abs(out['steered'] - out['plain']).max())
print('max |logit steered - plain| at init:', diff)
assert np.isfinite(out['steered']).all() and diff < 1e-3
del sd; print('PRE-FLIGHT OK')

In [ ]:
# MAIN EXPERIMENT. Safe to re-run after a disconnect. Each finished run evaluates the held-out test exactly once.
for seed in SEEDS:
    for arm in E.ARM_ORDER:
        E.run_one(E.m9_config(arm, seed), build_model, train_ds, val_ds, test_ds, RUNS, device=dev)
        gc.collect(); torch.cuda.empty_cache()

In [ ]:
# GATE + COMPARISON (no GPU work). Primary = final epoch; secondary = best-validation epoch.
gate = E.m9_gate(RUNS, SEEDS)
gate['comparison_final'] = E.compare(RUNS, SEEDS, which='final')
gate['comparison_best_val'] = E.compare(RUNS, SEEDS, which='best_val')
for k in ('comparison_final', 'comparison_best_val'):
    print(E.format_comparison(gate[k]), '\n')
print('frozen-CLIP probes on the same test set (M2, R001-R002): univfd 0.829 [0.802, 0.853], multilayer 0.868 [0.846, 0.891]')

In [ ]:
# DIAGNOSTICS (validation only): did training converge, and did steering actually do anything?
for seed in SEEDS:
    for arm in E.ARM_ORDER:
        rdir = f'{RUNS}/{E.run_name(E.m9_config(arm, seed))}'
        r, d = json.load(open(f'{rdir}/results.json')), json.load(open(f'{rdir}/val_diagnostics.json'))
        print(f'=== {arm} seed {seed}')
        for h in r['history']:
            print(f"  epoch {h['epoch']}: train loss {h['train_loss']:.4f} | val loss {h['val_loss']:.4f} | val AUROC {h['val_auroc']:.4f}")
        print(f"  alpha entropy {d['alpha']['entropy_mean']:.3f} of {d['alpha']['entropy_max_possible']:.3f}, "
              f"mean max alpha {d['alpha']['max_alpha_mean']:.3f}")
        print(f"  concept score mean: fake-oriented {d['readout']['score_mean_fake_oriented']:+.3f}, "
              f"real-oriented {d['readout']['score_mean_real_oriented']:+.3f}")
        if arm == 'steered':
            for g in d['gate_deviation']:
                print(f"  gates layer {g['layer']} |g-1| mean/max: " +
                      ', '.join(f"{p} {g[p+'_mean']:.4f}/{g[p+'_max']:.3f}" for p in ('dt', 'B', 'C')))
            print('  same weights, gates off:', d['gates_off'], '| AUROC gates on', round(d['auroc'], 4))

In [ ]:
# SAVE: gate JSON, run artefacts (no checkpoints) and experiment-log rows, into Drive and the repo clone.
files = E.export(RUNS, '/content/csm-clip/runs/m9', SEEDS)
rows = E.log_rows(RUNS, SEEDS)
for p in (f'{DRIVE_ROOT}/runs/m9_gate.json', '/content/csm-clip/runs/m9_gate.json'):
    json.dump(gate, open(p, 'w'), indent=1)
open('/content/csm-clip/runs/m9/experiment_log_rows.md', 'w').write(rows + '\n')
print('GATE', gate['gate'], '|', len(files), 'files in runs/m9/\n')
print(rows)
import shutil
shutil.make_archive('/content/m9_results', 'zip', '/content/csm-clip/runs', 'm9')
from google.colab import files as colab_files
colab_files.download('/content/m9_results.zip'); colab_files.download('/content/csm-clip/runs/m9_gate.json')
# Then: commit runs/m9/ and runs/m9_gate.json, paste the rows into docs/experiment_log.md,
# set M9 to Done in docs/contract.md, tag m9-done.